In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import StratifiedKFold, cross_val_score

%load_ext autoreload
%autoreload 2

In [2]:
df = pd.read_csv("data/train.csv")

df['Pclass'] = df['Pclass'].astype(str)


feature_columns = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
]

X = df[feature_columns].copy()
y = df["Survived"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_valid.shape

((712, 7), (179, 7))

## Data preprocessing 

In [3]:
numeric_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
categorical_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(sparse_output=False, handle_unknown="ignore"))


preprocessor = ColumnTransformer(transformers=[
    ('numeric_pipeline', numeric_pipeline, make_column_selector(dtype_include='number')),
    ('categorical_pipeline', categorical_pipeline, make_column_selector(dtype_include=['object', 'category']))
])



## Baseline model with dummy classifier
Baseline accuracy: 0.615


In [4]:

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_valid)

baseline_accuracy = accuracy_score(y_valid, baseline_predictions)
print(f"Baseline accuracy: {baseline_accuracy:.3f}")


Baseline accuracy: 0.615


## Logistic Regression

*accuracy:*      0.804469 </br>
*precision:*     0.793103 </br>
*recall:*        0.666667 </br>
*f1-score:*      0.724409 </br>
*support:*      69.000000 </br>

In [5]:
model_LogisticRegression = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))

model_LogisticRegression.fit(X_train, y_train)

predictions = model_LogisticRegression.predict(X_valid)
predictions_accuracy = accuracy_score(y_valid, predictions)
print(y_valid.mean())
print(predictions_accuracy)

print(confusion_matrix(y_valid, predictions))
class_report = classification_report(y_valid, predictions, output_dict=True)
print('логистическая регрессия')
pd.DataFrame(class_report)['1'] 

0.3854748603351955
0.8044692737430168
[[98 12]
 [23 46]]
логистическая регрессия


precision     0.793103
recall        0.666667
f1-score      0.724409
support      69.000000
Name: 1, dtype: float64

## StratifiedKFold with 5 folds

scores: [0.78212291 0.80337079 0.79775281 0.78089888 0.82022472] <br>
mean: 0.7968740192078337

In [6]:
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_LogisticRegression, X, y, cv=cv_stratified, scoring="accuracy")
print(scores)
print(scores.mean())
print(scores.std())

[0.78212291 0.80337079 0.79775281 0.78089888 0.82022472]
0.7968740192078337
0.01456733331525481


## StratifiedKFold with 5 folds without scaler

Модель без использование скейлера предсказывает на 0.1 %

In [7]:
numeric_pipeline_with_unscaled = make_pipeline(SimpleImputer(strategy="median"))


preprocessor_unscaled = ColumnTransformer(transformers=[
    ('numeric_pipeline_with_scaler', numeric_pipeline_with_unscaled, make_column_selector(dtype_include='number')),
    ('categorical_pipeline', categorical_pipeline, make_column_selector(dtype_include=['object', 'category']))
])

model_LogisticRegression_unscaled = make_pipeline(preprocessor_unscaled, LogisticRegression(max_iter=1000))

scores_unscaled = cross_val_score(model_LogisticRegression_unscaled, X, y, cv=cv_stratified, scoring="accuracy")
print(scores_unscaled)
print(scores_unscaled.mean())
print(scores_unscaled.std())

[0.7877095  0.80337079 0.79775281 0.78089888 0.82022472]
0.797991337643588
0.01357340849687361


In [8]:
df_experiments = pd.read_csv("experiments/experiments.csv")
df_experiments

,datetime,name,model,model_params,config,accuracy_mean,accuracy_std,roc_auc_mean,roc_auc_std
0,2026-09-24T17:12:31,baseline,LogisticRegression,"{""C"": 1.0, ""class_weight"": null, ""dual"": false...","{""features"": {""Age"": ""numeric"", ""Fare"": ""numer...",0.792398,0.018751,0.851905,0.021477
1,2026-09-24T17:12:35,pclass_onehot,LogisticRegression,"{""C"": 1.0, ""class_weight"": null, ""dual"": false...","{""features"": {""Age"": ""numeric"", ""Fare"": ""numer...",0.797991,0.013573,0.851455,0.021981
2,2026-09-24T17:13:08,family_size,LogisticRegression,"{""C"": 1.0, ""class_weight"": null, ""dual"": false...","{""features"": {""Age"": ""numeric"", ""Sex"": ""binary...",0.797991,0.009124,0.856340,0.018878
3,2026-09-24T17:13:27,family_size,LogisticRegression,"{""C"": 1.0, ""class_weight"": null, ""dual"": false...","{""features"": {""Age"": ""numeric"", ""Sex"": ""binary...",0.810332,0.008087,0.864090,0.015746
4,2026-09-24T17:13:40,family_size,LogisticRegression,"{""C"": 1.0, ""class_weight"": null, ""dual"": false...","{""features"": {""Age"": ""numeric"", ""Sex"": ""onehot...",0.810332,0.008087,0.863983,0.015725
...,...,...,...,...,...,...,...,...,...
96,2026-09-26T16:44:28,DecisionTreeClassifier feature selection,DecisionTreeClassifier,"{""ccp_alpha"": 0.0, ""class_weight"": null, ""crit...","{""features"": {""Age"": ""numeric"", ""Fare"": ""numer...",0.840625,0.018350,0.846336,0.022226
97,2026-09-26T16:44:40,DecisionTreeClassifier feature selection,DecisionTreeClassifier,"{""ccp_alpha"": 0.0, ""class_weight"": null, ""crit...","{""features"": {""Age"": ""numeric"", ""Fare"": ""numer...",0.835020,0.017946,0.842299,0.022599
98,2026-09-26T16:44:47,DecisionTreeClassifier feature selection,DecisionTreeClassifier,"{""ccp_alpha"": 0.0, ""class_weight"": null, ""crit...","{""features"": {""Age"": ""numeric"", ""Fare"": ""numer...",0.839502,0.018014,0.844038,0.022446
99,2026-09-26T16:44:52,DecisionTreeClassifier feature selection,DecisionTreeClassifier,"{""ccp_alpha"": 0.0, ""class_weight"": null, ""crit...","{""features"": {""Age"": ""numeric"", ""Fare"": ""numer...",0.838391,0.018534,0.845186,0.021988


## Logistic Regression — experiments

Все эксперименты сравниваются на одинаковом **5-fold StratifiedKFold**
(`shuffle=True`, `random_state=42`).

Основная метрика для выбора модели — **accuracy**. ROC-AUC используем как дополнительную.


In [9]:
from sklearn_experiments import evaluate_model, base_config, tune_model
from sklearn.linear_model import LogisticRegression
import copy

df_train = pd.read_csv("data/train.csv")

model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

baseline_result = evaluate_model(
    df=df_train,
    target="Survived",
    model=model,
    config=base_config,
)

baseline_result


{'accuracy_mean': np.float64(0.7923984683949532),
 'accuracy_std': np.float64(0.018750529151998523),
 'roc_auc_mean': np.float64(0.8519052001988017),
 'roc_auc_std': np.float64(0.021477125487472243)}

### Feature engineering — итог

Проверили обработку категорий и дополнительные признаки.

**Полезными оказались:** `Pclass` как one-hot, `FareLog1p`, `SexPclass`, `AgeSquared`, `IsAlone`.

**Не дали заметной пользы:** `FamilySize`, смена encoding для `Sex`, дополнительный scaler.

`IsChild` ухудшил среднюю accuracy.

Лучший набор признаков дал:

- **accuracy: 0.8159 ± 0.0081**
- **ROC-AUC: 0.8640 ± 0.0122**


In [10]:
logreg_best_config = {
    "features": {
        "Age": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "onehot",
        "_Fare": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },

    "fare_log1p": True,
    "sex_pclass": True,
    "age_squared": True,
    "is_alone": True,

    "is_child": False,
    "family_size": False,
    "scale_numeric": True,
}

best_feature_result = evaluate_model(
    df=df_train,
    target="Survived",
    model=model,
    config=logreg_best_config,
)

best_feature_result


{'accuracy_mean': np.float64(0.8159437574540205),
 'accuracy_std': np.float64(0.008091836959853813),
 'roc_auc_mean': np.float64(0.8639580262608725),
 'roc_auc_std': np.float64(0.012196395216648746)}

## Regularization and hyperparameter tuning

Проверяем:

- `C`: силу регуляризации;
- `l1_ratio = 0` — L2;
- `l1_ratio = 1` — L1;
- промежуточные значения — ElasticNet.

Для ElasticNet используем `solver="saga"`.


In [11]:
tuned_model = LogisticRegression(
    solver="saga",
    max_iter=1000,
    random_state=42,
)

param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0],
}

grid_features = tune_model(
    df=df_train,
    target="Survived",
    model=tuned_model,
    param_grid=param_grid,
    config=logreg_best_config,
)

print("Best accuracy:", grid_features.best_score_)
print("Best params:", grid_features.best_params_)
print(
    "Iterations:",
    grid_features.best_estimator_.named_steps["model"].n_iter_,
)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best accuracy: 0.8159437574540205
Best params: {'model__C': 1, 'model__l1_ratio': 0.0}
Iterations: [187]


In [12]:
results = pd.DataFrame(grid_features.cv_results_)

results[
    [
        "param_model__C",
        "param_model__l1_ratio",
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_test_roc_auc",
        "std_test_roc_auc",
    ]
].sort_values(
    "mean_test_accuracy",
    ascending=False,
).head(10)


,param_model__C,param_model__l1_ratio,mean_test_accuracy,std_test_accuracy,mean_test_roc_auc,std_test_roc_auc
15,1.0,0.00,0.815944,0.008092,0.863932,0.012152
21,10.0,0.25,0.815937,0.006529,0.863053,0.012510
22,10.0,0.50,0.815937,0.006529,0.863105,0.012754
23,10.0,0.75,0.815937,0.006529,0.863131,0.012709
24,10.0,1.00,0.815937,0.006529,0.863105,0.012569
19,1.0,1.00,0.814820,0.009939,0.863791,0.013080
20,10.0,0.00,0.814814,0.007956,0.863026,0.012462
25,100.0,0.00,0.814814,0.007956,0.863104,0.012737
27,100.0,0.50,0.814814,0.007956,0.863184,0.012682
26,100.0,0.25,0.814814,0.007956,0.863158,0.012702


### Regularization — вывод

Лучший результат: **C=1, L2 (`l1_ratio=0`)**.

- accuracy: **0.8159**
- модель сошлась за **187 итераций**, поэтому `max_iter=1000` достаточно;
- L1 и ElasticNet улучшения не дали;
- подбор регуляризации не улучшил результат относительно лучшего feature engineering.


## Проверка: tuning или feature engineering?

Чтобы понять источник прироста, запускаем тот же GridSearch на базовых признаках без feature engineering.


In [13]:
grid_base = tune_model(
    df=df_train,
    target="Survived",
    model=tuned_model,
    param_grid=param_grid,
    config=base_config,
)

print("Tuned base features:", grid_base.best_score_)
print("Best params:", grid_base.best_params_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Tuned base features: 0.8024794425961961
Best params: {'model__C': 0.1, 'model__l1_ratio': 0.0}


## Logistic Regression — финальный итог

| Вариант | CV accuracy |
| --- | ---: |
| Базовая Logistic Regression | **0.7924** |
| Базовые признаки + tuning | **0.8025** |
| Feature engineering + tuning | **0.8159** |

**Вывод:** tuning сам по себе помог, но feature engineering дал дополнительный прирост примерно **+1.35 п.п.** относительно уже настроенной модели.

Лучший вариант: **feature engineering + LogisticRegression, C=1, L2**.


## KNN

Проверили три шага:

1. базовый KNN на исходных признаках;
2. подбор `n_neighbors`, `weights`, `metric`;
3. feature engineering / feature selection для KNN.

Во всех экспериментах числовые признаки масштабируются.


In [14]:
from sklearn.neighbors import KNeighborsClassifier

knn_base_config = {
    "features": {
        "Age": "numeric",
        "Fare": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },
    "scale_numeric": True,
}

knn_baseline = evaluate_model(
    df=df_train,
    target="Survived",
    model=KNeighborsClassifier(),
    config=knn_base_config,
)

knn_baseline


{'accuracy_mean': np.float64(0.8047266336074321),
 'accuracy_std': np.float64(0.02246132953142063),
 'roc_auc_mean': np.float64(0.84881001644593),
 'roc_auc_std': np.float64(0.026386557561435802)}

### Baseline

**KNN по умолчанию:**

- accuracy: **0.8047 ± 0.0225**
- ROC-AUC: **0.8488 ± 0.0264**


In [15]:
param_grid_knn = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}

grid_knn = tune_model(
    df=df_train,
    target="Survived",
    model=KNeighborsClassifier(),
    param_grid=param_grid_knn,
    config=knn_base_config,
)

print("Best accuracy:", grid_knn.best_score_)
print("Best params:", grid_knn.best_params_)


Fitting 5 folds for each of 28 candidates, totalling 140 fits
Best accuracy: 0.8148201619484026
Best params: {'model__metric': 'manhattan', 'model__n_neighbors': 21, 'model__weights': 'uniform'}


### Hyperparameter tuning

Лучшие параметры:

- `n_neighbors = 21`
- `weights = "uniform"`
- `metric = "manhattan"`

Accuracy на базовых признаках выросла до **0.8148**.


In [16]:
knn_best_config = {
    "features": {
        "Age": "numeric",
        "_Fare": "numeric",
        "Sex": "onehot",
        "Embarked": "onehot",
        "Pclass": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },
    "scale_numeric": True,
    "family_size": True,
    "is_alone": False,
    "is_child": False,
    "fare_log1p": False,
    "age_squared": False,
    "sex_pclass": False,
}

best_knn = KNeighborsClassifier(
    n_neighbors=21,
    weights="uniform",
    metric="manhattan",
)

knn_best_result = evaluate_model(
    df=df_train,
    target="Survived",
    model=best_knn,
    config=knn_best_config,
)

knn_best_result


{'accuracy_mean': np.float64(0.8316552633230808),
 'accuracy_std': np.float64(0.008585631966684338),
 'roc_auc_mean': np.float64(0.8582623218268705),
 'roc_auc_std': np.float64(0.023367825986457164)}

### KNN — итог

| Вариант | CV accuracy |
| --- | ---: |
| KNN baseline | **0.8047** |
| KNN + tuning | **0.8148** |
| KNN + tuning + feature engineering | **0.8317** |

Лучший вариант:

- `n_neighbors=21`
- `weights="uniform"`
- `metric="manhattan"`
- `Sex` → one-hot
- `Pclass` оставлен числовым
- добавлен `FamilySize`
- `Fare` исключён
- scaling включён

**Лучший результат: accuracy = 0.8317 ± 0.0086, ROC-AUC = 0.8583 ± 0.0234.**

Что показали эксперименты:

- scaling для KNN критичен: без него качество падало примерно до **0.781**;
- `FamilySize` и представление `Sex` через one-hot помогли;
- исключение `Fare` улучшило лучший вариант;
- добавление сразу всех engineered features результата не улучшило.


## Decision Tree

Проверили:

1. дерево без ограничений;
2. переобучение по разнице train / CV;
3. ограничение сложности через GridSearch;
4. небольшой feature engineering.


In [17]:
from sklearn.tree import DecisionTreeClassifier

tree_config = {
    "features": {
        "Age": "numeric",
        "Fare": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },
    "scale_numeric": False,
}

dt_baseline = evaluate_model(
    df=df_train,
    target="Survived",
    model=DecisionTreeClassifier(random_state=42),
    config=tree_config,
)

dt_baseline


{'accuracy_mean': np.float64(0.7878789780930262),
 'accuracy_std': np.float64(0.03105183159581353),
 'roc_auc_mean': np.float64(0.7822737031498969),
 'roc_auc_std': np.float64(0.03154599971106935)}

### Baseline и переобучение

Неограниченное дерево:

- **CV accuracy: 0.7879 ± 0.0311**
- **train accuracy: 0.9798**

Большой разрыв между train и CV показывает сильное переобучение.


In [18]:
param_grid_dt = {
    "model__max_depth": [7, 8, 9],
    "model__min_samples_split": [8, 10, 12, 15],
    "model__min_samples_leaf": [1, 2, 3],
    "model__criterion": ["gini", "entropy"],
}

grid_dt = tune_model(
    df=df_train,
    target="Survived",
    model=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_dt,
    config=tree_config,
)

print("Best accuracy:", grid_dt.best_score_)
print("Best params:", grid_dt.best_params_)


Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best accuracy: 0.8383780051471973
Best params: {'model__criterion': 'gini', 'model__max_depth': 8, 'model__min_samples_leaf': 1, 'model__min_samples_split': 10}


### Hyperparameter tuning

Лучшие параметры:

- `criterion="gini"`
- `max_depth=8`
- `min_samples_split=10`
- `min_samples_leaf=1`

После ограничения сложности:

- **CV accuracy: 0.8384**
- **train accuracy: 0.8822**

Разрыв train / CV заметно уменьшился, а качество на CV выросло.


In [19]:
tree_best_config = {
    "features": {
        "Age": "numeric",
        "Fare": "numeric",
        "Sex": "binary",
        "Embarked": "onehot",
        "Pclass": "numeric",
        "SibSp": "numeric",
        "Parch": "numeric",
    },
    "scale_numeric": False,
    "sex_pclass": True,
}

best_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=1,
    random_state=42,
)

tree_best_result = evaluate_model(
    df=df_train,
    target="Survived",
    model=best_tree,
    config=tree_best_config,
)

tree_best_result


{'accuracy_mean': np.float64(0.8428661100998054),
 'accuracy_std': np.float64(0.020149654339052248),
 'roc_auc_mean': np.float64(0.8471923924953091),
 'roc_auc_std': np.float64(0.021245244391572174)}

### Decision Tree — итог

| Вариант | CV accuracy |
| --- | ---: |
| Без ограничений | **0.7879** |
| После tuning | **0.8384** |
| Tuning + `SexPclass` | **0.8429** |

Лучший результат:

- **accuracy = 0.8429 ± 0.0202**
- **ROC-AUC = 0.8472 ± 0.0212**

Главный вывод: неограниченное дерево сильно переобучалось. Ограничение глубины и минимального размера узла перед split существенно улучшило обобщение. Из проверенного feature engineering лучший результат дал `SexPclass`.
